# Boulevard Boosters Demo

This notebook demonstrates the sklearn-compatible release API:

- `bd.DropoutBooster`
- `bd.ParallelBooster`
- `bd.ExplainableBooster`

Recommended first-use pattern:

1. Use `DropoutBooster` or `ParallelBooster` for low-dimensional smooth regression signals.
2. Use `ExplainableBooster` as the first option when dimension grows and an additive/main-effect model is acceptable.
3. Fit the model, call `prepare_inference`, then call `predict` or `predict_intervals`.

The intervals here are asymptotic research intervals. Prediction intervals are usually more stable than signal confidence intervals because they include residual noise variance.


In [ ]:
import os
import sys
import time
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/boulevard-matplotlib-cache")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
project_root = Path.cwd()
if not (project_root / "src" / "boulevard").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
from sklearn.metrics import mean_squared_error  # noqa: E402
from sklearn.model_selection import train_test_split  # noqa: E402

import boulevard as bd  # noqa: E402


In [ ]:
def rmse(target, pred):
    return float(np.sqrt(mean_squared_error(target, pred)))


def coverage(target, lower, upper):
    return float(np.mean((target >= lower) & (target <= upper)))


def width_quantiles(lower, upper):
    return np.quantile(upper - lower, [0.05, 0.5, 0.95])


def split_train_calib_test(X, y, signal, *, seed=0):
    X_train, X_hold, y_train, y_hold, _signal_train, signal_hold = train_test_split(
        X,
        y,
        signal,
        test_size=0.4,
        random_state=seed,
    )
    X_calib, X_test, y_calib, y_test, _signal_calib, signal_test = train_test_split(
        X_hold,
        y_hold,
        signal_hold,
        test_size=0.5,
        random_state=seed + 1,
    )
    return X_train, X_calib, X_test, y_train, y_calib, y_test, signal_test


def timed_fit_prepare(model, X_train, y_train, X_calib, y_calib):
    start = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - start

    start = time.perf_counter()
    model.prepare_inference(X_calib, y_calib)
    prep_seconds = time.perf_counter() - start
    return fit_seconds, prep_seconds


def evaluate_model(model, X_test, y_test, signal_test):
    start = time.perf_counter()
    pred = model.predict(X_test)
    pred_seconds = time.perf_counter() - start

    intervals = {}
    interval_seconds = {}
    for mode in ["confidence", "prediction", "reproduction"]:
        start = time.perf_counter()
        intervals[mode] = model.predict_intervals(X_test, level=0.95, mode=mode)
        interval_seconds[mode] = time.perf_counter() - start

    ci_lower, ci_upper, _ = intervals["confidence"]
    pi_lower, pi_upper, _ = intervals["prediction"]
    return {
        "pred": pred,
        "intervals": intervals,
        "pred_seconds": pred_seconds,
        "interval_seconds": interval_seconds,
        "rmse_signal": rmse(signal_test, pred),
        "rmse_y": rmse(y_test, pred),
        "ci_coverage": coverage(signal_test, ci_lower, ci_upper),
        "pi_coverage": coverage(y_test, pi_lower, pi_upper),
        "ci_width_quantiles": width_quantiles(ci_lower, ci_upper),
    }


## Low-Dimensional Example: DropoutBooster and ParallelBooster

For one-dimensional or otherwise very low-dimensional smooth signals, the histogram-tree BRAT estimators are the main examples. This section fits both estimators on the same sinusoidal regression problem and compares confidence, prediction, and reproduction intervals.


In [ ]:
rng = np.random.default_rng(0)


def signal_1d(X):
    return np.sin(2 * np.pi * X[:, 0])


X_1d = rng.uniform(0.0, 1.0, size=(1000, 1))
signal_1d_values = signal_1d(X_1d)
y_1d = signal_1d_values + rng.normal(scale=0.2, size=X_1d.shape[0])

X1_train, X1_calib, X1_test, y1_train, y1_calib, y1_test, signal1_test = (
    split_train_calib_test(X_1d, y_1d, signal_1d_values)
)

low_dim_models = {
    "DropoutBooster": bd.DropoutBooster(
        max_iter=700,
        learning_rate=0.8,
        dropout_rate=0.1,
        subsample_rate=0.8,
        max_depth=6,
        max_leaf_nodes=64,
        min_samples_leaf=2,
        max_bins=64,
        early_stopping=False,
        random_state=0,
    ),
    "ParallelBooster": bd.ParallelBooster(
        n_rounds=70,
        trees_per_round=6,
        subsample_rate=0.8,
        max_depth=8,
        max_leaf_nodes=128,
        min_samples_leaf=8,
        max_bins=64,
        drop_first_round=True,
        early_stopping=False,
        n_jobs=1,
        random_state=0,
    ),
}

low_dim_results = {}
for name, model in low_dim_models.items():
    fit_seconds, prep_seconds = timed_fit_prepare(
        model,
        X1_train,
        y1_train,
        X1_calib,
        y1_calib,
    )
    metrics = evaluate_model(model, X1_test, y1_test, signal1_test)
    metrics["fit_seconds"] = fit_seconds
    metrics["prep_seconds"] = prep_seconds
    low_dim_results[name] = metrics

for name, metrics in low_dim_results.items():
    print(f"\n{name}")
    print(f"  fit seconds: {metrics['fit_seconds']:.3f}")
    print(f"  prepare_inference seconds: {metrics['prep_seconds']:.3f}")
    print(f"  predict seconds: {metrics['pred_seconds']:.4f}")
    print(f"  RMSE vs signal: {metrics['rmse_signal']:.4f}")
    print(f"  RMSE vs noisy y: {metrics['rmse_y']:.4f}")
    print(f"  signal CI coverage: {metrics['ci_coverage']:.3f}")
    print(f"  noisy-y PI coverage: {metrics['pi_coverage']:.3f}")
    print(f"  CI width q05/q50/q95: {metrics['ci_width_quantiles']}")


In [ ]:
grid = np.linspace(0.0, 1.0, 220)
X_grid_1d = grid.reshape(-1, 1)
truth_grid_1d = signal_1d(X_grid_1d)
mode_colors = {
    "confidence": "#1f77b4",
    "prediction": "#d62728",
    "reproduction": "#2ca02c",
}
mode_labels = {"confidence": "CI", "prediction": "PI", "reproduction": "RI"}

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True, sharey=True)
for row, name in enumerate(["DropoutBooster", "ParallelBooster"]):
    model = low_dim_models[name]
    for col, mode in enumerate(["confidence", "prediction", "reproduction"]):
        lower, upper, pred = model.predict_intervals(X_grid_1d, level=0.95, mode=mode)
        ax = axes[row, col]
        ax.scatter(
            X1_train[:, 0],
            y1_train,
            s=8,
            color="0.75",
            alpha=0.25,
            label="train y",
        )
        ax.fill_between(grid, lower, upper, color=mode_colors[mode], alpha=0.18)
        ax.plot(grid, truth_grid_1d, color="black", linewidth=1.8, label="signal")
        ax.plot(grid, pred, color=mode_colors[mode], linewidth=1.8, label="fit")
        ax.set_title(f"{name} {mode_labels[mode]}")
        ax.set_xlabel("x")
        ax.set_ylabel("response")
axes[0, 0].legend(loc="best")
fig.tight_layout()


## Higher-Dimensional Additive Example: ExplainableBooster

When dimension grows, start with `ExplainableBooster` if an additive/main-effect model is scientifically acceptable. It gives feature-level intervals and usually avoids the high-dimensional cell explosion that makes global histogram-tree signal intervals fragile.


In [ ]:
def signal_additive(X):
    return (
        np.sin(2 * np.pi * X[:, 0])
        + 0.8 * (X[:, 1] - 0.5)
        + 0.4 * np.cos(4 * np.pi * X[:, 2])
    )


def feature_truth(feature_idx, values):
    if feature_idx == 0:
        return np.sin(2 * np.pi * values)
    if feature_idx == 1:
        return 0.8 * (values - 0.5)
    if feature_idx == 2:
        return 0.4 * np.cos(4 * np.pi * values)
    raise ValueError(feature_idx)


X_add = rng.uniform(0.0, 1.0, size=(2500, 3))
signal_add = signal_additive(X_add)
y_add = signal_add + rng.normal(scale=0.25, size=X_add.shape[0])

(
    X_train,
    X_calib,
    X_test,
    y_train,
    y_calib,
    y_test,
    signal_test,
) = split_train_calib_test(X_add, y_add, signal_add)

explainable = bd.ExplainableBooster(
    max_rounds=160,
    max_bins=32,
    learning_rate=0.6,
    subsample_rate=1.0,
    warmup_rounds=10,
    max_depth=4,
    min_samples_leaf=8,
    random_state=0,
)
fit_seconds, prep_seconds = timed_fit_prepare(
    explainable,
    X_train,
    y_train,
    X_calib,
    y_calib,
)
explainable_metrics = evaluate_model(explainable, X_test, y_test, signal_test)

print("ExplainableBooster")
print(f"  fit seconds: {fit_seconds:.3f}")
print(f"  prepare_inference seconds: {prep_seconds:.3f}")
print(f"  predict seconds: {explainable_metrics['pred_seconds']:.4f}")
print(f"  RMSE vs signal: {explainable_metrics['rmse_signal']:.4f}")
print(f"  RMSE vs noisy y: {explainable_metrics['rmse_y']:.4f}")
print(f"  signal CI coverage: {explainable_metrics['ci_coverage']:.3f}")
print(f"  noisy-y PI coverage: {explainable_metrics['pi_coverage']:.3f}")
print(f"  CI width q05/q50/q95: {explainable_metrics['ci_width_quantiles']}")


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 10), sharex=True)
for feature_idx in range(3):
    centered_truth = feature_truth(feature_idx, grid) - np.mean(
        feature_truth(feature_idx, X_train[:, feature_idx])
    )
    for col, mode in enumerate(["confidence", "prediction", "reproduction"]):
        lower, upper, pred = explainable.predict_feature_intervals(
            feature_idx,
            grid,
            level=0.95,
            mode=mode,
            include_intercept=False,
        )
        feature_coverage = coverage(centered_truth, lower, upper)
        ax = axes[feature_idx, col]
        ax.fill_between(grid, lower, upper, color=mode_colors[mode], alpha=0.18)
        ax.plot(grid, centered_truth, color="black", linewidth=1.8, label="truth")
        ax.plot(grid, pred, color=mode_colors[mode], linewidth=1.8, label="partial fit")
        ax.set_title(
            f"ExplainableBooster x{feature_idx} {mode_labels[mode]} "
            f"coverage={feature_coverage:.3f}"
        )
        ax.set_xlabel(f"x{feature_idx}")
        ax.set_ylabel("centered contribution")
axes[0, 0].legend(loc="best")
fig.tight_layout()


## Practical Recommendation

- Use `DropoutBooster` and `ParallelBooster` first for low-dimensional smooth regression problems.
- As dimension increases, increase sample size and model capacity, then validate signal-CI coverage on a problem-specific diagnostic.
- For higher-dimensional additive structure, prefer `ExplainableBooster` because it exposes feature-level intervals and keeps the inference problem lower-dimensional.
- For interaction-heavy high-dimensional functions, treat signal confidence intervals as diagnostic. Prediction intervals are currently the safer interval type for outcome uncertainty.
